# Notes
Helpful info from udemy course: 
https://github.com/mrdbourke/pytorch-deep-learning/blob/main/01_pytorch_workflow.ipynb

# GitHub Repository:
https://github.com/mpennino/Future_DW_NO3

In [506]:
# Import libraries
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import random


import pyarrow as pa
import pyarrow.parquet as pq


In [507]:
# Make device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [508]:
# Load Observation Dataset
# DATA = readRDS(paste0(strap_dir,'Data/Models/RF_bi_model_All_DATA_all_vars_','Trends_Conc_PWS_GW_05to20', '.rds'))
#future_dir = 'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/StRAPs/StRAP4/SSWR.405.1_Future_DW/Data/'
future_dir = 'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/OASES/Data/Future_NO3/'


dataset1 = 'Dataset_RF_Model_SW_COMIDS.csv'
#dataset1 = 'Dataset_RF_Model_GW_COMIDS.csv'

input_data_ = pd.read_csv(future_dir+dataset1)
input_data_.head(3)

,COMID,viol_freq,PopDen2010Ws,PctForest2019Ws,PctCrop2019Ws,precip9120ws,tmean9120ws,BFIWs,permws,N_TW2012Ws,RockNWs,N_Surp_kgsqkm_2017ws,ElevWs,Fe2O3Ws,NHDslope_Pct_Ws,Viol_Class
0,12558,0,21.5842,40.60,7.02,446.418385,9.098713,64.9434,7.301968,3.639743,413.0962,1281.176035,1745.8597,2.7018,9.338324,0
1,12564,0,4.8193,63.36,0.00,454.415442,9.029232,65.0000,8.889206,3.607679,422.7846,350.849192,1866.1304,3.6657,15.421640,0
2,12606,0,1.1832,68.73,0.00,678.346321,2.958328,69.1535,14.232988,4.387261,27.3868,492.682789,2905.9906,2.5098,8.917415,0


In [509]:
input_data_.shape


(14934, 16)

In [510]:
input_data1 = input_data_.drop(columns=['viol_freq']) 

names_list = input_data1.columns.tolist()
print(names_list) 

['COMID', 'PopDen2010Ws', 'PctForest2019Ws', 'PctCrop2019Ws', 'precip9120ws', 'tmean9120ws', 'BFIWs', 'permws', 'N_TW2012Ws', 'RockNWs', 'N_Surp_kgsqkm_2017ws', 'ElevWs', 'Fe2O3Ws', 'NHDslope_Pct_Ws', 'Viol_Class']


In [511]:
# Remove extra fields
# For Surface Water Dataset (,'AgDrain_pctWs','Hillslope_PctWs','BFIWs')
#input_data = input_data_.drop(columns=['HUC12','viol_freq','PopDen2010Ws','WaterInputWs','wdrw_LDWs','FertWs','CBNFWs','ManureWs','Septic_km2Cat']) 
input_data = input_data_.drop(columns=['COMID','viol_freq']) 

# For Groundwater Dataset
#input_data = input_data_.drop(columns=['HUC12','viol_freq','PopDen2010Cat','AgKffactCat','Septic_km2Cat','AgDrain_pctCat','WaterInputCat','wdrw_LDCat','BFICat','Hillslope_PctCat']) 

input_data.head(3)


,PopDen2010Ws,PctForest2019Ws,PctCrop2019Ws,precip9120ws,tmean9120ws,BFIWs,permws,N_TW2012Ws,RockNWs,N_Surp_kgsqkm_2017ws,ElevWs,Fe2O3Ws,NHDslope_Pct_Ws,Viol_Class
0,21.5842,40.60,7.02,446.418385,9.098713,64.9434,7.301968,3.639743,413.0962,1281.176035,1745.8597,2.7018,9.338324,0
1,4.8193,63.36,0.00,454.415442,9.029232,65.0000,8.889206,3.607679,422.7846,350.849192,1866.1304,3.6657,15.421640,0
2,1.1832,68.73,0.00,678.346321,2.958328,69.1535,14.232988,4.387261,27.3868,492.682789,2905.9906,2.5098,8.917415,0


In [512]:
input_data.shape,input_data_.shape

((14934, 14), (14934, 16))

In [513]:
# Calculate accuracy (a classification metric)
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item() # torch.eq() calculates where two tensors are equal
    acc = (correct / len(y_pred)) * 100 
    return acc

# Create Balanced Dataset


In [514]:
print(input_data['Viol_Class'].value_counts())

Viol_Class
0    14893
1       41
Name: count, dtype: int64


In [515]:
# Save Preditor Data for SHAP Analysis
X_input_data = input_data.drop(columns=['Viol_Class']).values
X_input_data.shape


(14934, 13)

In [516]:
# Convert to tensor data
X_input_data = torch.from_numpy(X_input_data).type(torch.float)
X_input_data.shape,X_input_data.dtype

(torch.Size([14934, 13]), torch.float32)

In [517]:
min_size = input_data['Viol_Class'].value_counts().min()
min_size

np.int64(41)

In [518]:
# Find the size of the smallest class
min_size = input_data['Viol_Class'].value_counts().min()

min_size  = min_size * 10

# Sample exactly 'min_size' elements from each binary group
#balanced_df = input_data.groupby('Viol_Class').sample(n=min_size, random_state=42).reset_index(drop=True)

# If increasing the min_size, you can use the 'replace=True' argument to allow for sampling with replacement
balanced_df = input_data.groupby('Viol_Class').sample(n=min_size, random_state=42, replace=True).reset_index(drop=True)

print(balanced_df['Viol_Class'].value_counts())

Viol_Class
0    410
1    410
Name: count, dtype: int64


# Transform data to torch tensor


In [519]:
# Convert to tensors and split into train and test sets
from sklearn.model_selection import train_test_split
#X = input_data.drop(columns=['Viol_Class']).values # when use this the model just predicts the majority class
#y = input_data['Viol_Class'].values
X = balanced_df.drop(columns=['Viol_Class']).values
y = balanced_df['Viol_Class'].values

# Turn data into tensors
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)

# Make a copy to use later
X_full = X.clone()
y_full = y.clone()

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2,
                                                    random_state=2
)

#X_train[:5], y_train[:5]

In [520]:
# print(y_train.unique(return_counts=True)),
# print(y_test.unique(return_counts=True)),

In [521]:
#X.dtype, y.dtype, X.size(), y.size()

# Create NN Model

In [522]:
nrows = X_train.size()[0]
ncols = X_train.size()[1]
nrows,ncols

(656, 13)

In [523]:
# Build SW model with non-linear activation function
# from torch import nn

# featureNum = 5
# class BinaryClassifier(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.layer_1 = nn.Linear(in_features=ncols, out_features=featureNum) 
#         self.layer_2 = nn.Linear(in_features=featureNum, out_features=featureNum)
#         self.layer_3 = nn.Linear(in_features=featureNum, out_features=1)
#         self.relu = nn.ReLU() # <- add in ReLU activation function (for non-linearity)
#         #self.relu= nn.leakyReLU() # <- add in ReLU activation function (for non-linearity)
#         # Can also put sigmoid in the model 
#         # This would mean you don't need to use it on the predictions
#         # self.sigmoid = nn.Sigmoid()

#     def forward(self, x):
#       # Intersperse the ReLU activation function between layers
#        return self.layer_3(self.relu(self.layer_2(self.relu(self.layer_1(x)))))

# model1 = BinaryClassifier().to(device)
# print(model1)

In [524]:
# # Build GW model with non-linear activation function
# from torch import nn

# featureNum = 5
# class BinaryClassifier(nn.Module):
#     def __init__(self):
#         super().__init__()
#         # This code works for SW HUC12
#         # self.layer_1 = nn.Linear(in_features=ncols, out_features=5) 
#         # self.layer_2 = nn.Linear(in_features=5, out_features=5)
#         # self.layer_3 = nn.Linear(in_features=5, out_features=1)

#         self.layer_1 = nn.Linear(in_features=ncols, out_features=featureNum) 
#         self.layer_2 = nn.Linear(in_features=featureNum, out_features=featureNum)
#         self.layer_3 = nn.Linear(in_features=featureNum, out_features=featureNum)
#         self.layer_4 = nn.Linear(in_features=featureNum, out_features=featureNum)
#         self.layer_5 = nn.Linear(in_features=featureNum, out_features=1)

#         self.relu = nn.ReLU(0.1) # <- add in ReLU activation function (for non-linearity)
#         #self.relu= nn.leakyReLU() # <- add in ReLU activation function (for non-linearity)
#         # Can also put sigmoid in the model 
#         # This would mean you don't need to use it on the predictions
#         # self.sigmoid = nn.Sigmoid()

#     def forward(self, x):
#       # Intersperse the ReLU activation function between layers
#        #return self.layer_3(self.relu(self.layer_2(self.relu(self.layer_1(x)))))
#        #return self.layer_4(self.layer_3(self.relu(self.layer_2(self.relu(self.layer_1(x))))))
#        return self.layer_5(self.layer_4(self.layer_3(self.relu(self.layer_2(self.relu(self.layer_1(x)))))))

# model1 = BinaryClassifier().to(device)
#print(model2)

In [525]:
import torch
import torch.nn as nn

class ImprovedBinaryClassifier(nn.Module):
    def __init__(self, input_dim=ncols, hidden_dim=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.1),             # Prevents dead neurons
            nn.BatchNorm1d(hidden_dim),     # Stabilizes training
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.2),                # Prevents overfitting
            
            nn.Linear(hidden_dim, 1)        # Outputs raw logits (No Sigmoid here!)
        )
        
    def forward(self, x):
        return self.network(x)

model1 = ImprovedBinaryClassifier().to(device)


In [526]:
# Setup loss and optimizer 
loss_fn = nn.BCEWithLogitsLoss()
#optimizer = torch.optim.SGD(model1.parameters(), lr=0.01)
optimizer1 = torch.optim.Adam(model1.parameters(), lr=0.01)
#optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.01)

# Train the Model

In [527]:
# Fit the model
# Set the random seed for PyTorch (CPU)
seedvalue = 24
torch.manual_seed(seedvalue)

# Set the random seed for PyTorch (GPU / CUDA) if you use a graphics card
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(seedvalue)

# Set the random seed for NumPy
np.random.seed(seedvalue)

# Set the random seed for Python's built-in random library
random.seed(seedvalue)

epochs = 1000

# Put all data on target device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

for epoch in range(epochs):
    # 1. Forward pass
    y_logits = model1(X_train).squeeze()
    y_pred = torch.round(torch.sigmoid(y_logits)) # logits -> prediction probabilities -> prediction labels
    
    # 2. Calculate loss and accuracy
    loss = loss_fn(y_logits, y_train) # BCEWithLogitsLoss calculates loss using logits
    acc = accuracy_fn(y_true=y_train, 
                      y_pred=y_pred)
    
    # 3. Optimizer zero grad
    optimizer1.zero_grad()

    # 4. Loss backward
    loss.backward()

    # 5. Optimizer step
    optimizer1.step()

    ### Testing
    model1.eval()
    with torch.inference_mode():
      # 1. Forward pass
      test_logits = model1(X_test).squeeze()
      test_pred = torch.round(torch.sigmoid(test_logits)) # logits -> prediction probabilities -> prediction labels
      # 2. Calculate loss and accuracy
      test_loss = loss_fn(test_logits, y_test)
      test_acc = accuracy_fn(y_true=y_test,
                             y_pred=test_pred)

    # Print out what's happening
    if epoch % 100 == 0:
        print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Test Loss: {test_loss:.5f}, Test Accuracy: {test_acc:.2f}%")

Epoch: 0 | Loss: 0.70937, Accuracy: 46.65% | Test Loss: 0.68186, Test Accuracy: 65.85%


Epoch: 100 | Loss: 0.21050, Accuracy: 91.77% | Test Loss: 0.24700, Test Accuracy: 87.20%
Epoch: 200 | Loss: 0.12337, Accuracy: 95.43% | Test Loss: 0.19313, Test Accuracy: 93.29%
Epoch: 300 | Loss: 0.13111, Accuracy: 94.51% | Test Loss: 0.17343, Test Accuracy: 95.73%
Epoch: 400 | Loss: 0.04763, Accuracy: 97.87% | Test Loss: 0.10530, Test Accuracy: 96.34%
Epoch: 500 | Loss: 0.12250, Accuracy: 95.58% | Test Loss: 0.23217, Test Accuracy: 89.02%
Epoch: 600 | Loss: 0.03175, Accuracy: 99.54% | Test Loss: 0.11816, Test Accuracy: 96.95%
Epoch: 700 | Loss: 0.01483, Accuracy: 99.85% | Test Loss: 0.11597, Test Accuracy: 96.95%
Epoch: 800 | Loss: 0.00770, Accuracy: 100.00% | Test Loss: 0.12778, Test Accuracy: 97.56%
Epoch: 900 | Loss: 0.00426, Accuracy: 100.00% | Test Loss: 0.14761, Test Accuracy: 97.56%


# Model Evaluation Metrics
*PCC, Sensativity, Specificity, AUC

In [528]:
import torchmetrics

# Define your classification task ('binary', 'multiclass', or 'multilabel')
task = "binary"

# Initialize metrics
sensitivity_metric = torchmetrics.classification.Recall(task=task)
specificity_metric = torchmetrics.classification.Specificity(task=task)

# Get model predictions on the test set
model1.eval()
with torch.inference_mode():
    preds = torch.round(torch.sigmoid(model1(X_test))).squeeze()

# get target / observed response values
target = y_test

# Simulated model predictions (logits or probabilities) and ground truth targets
# preds  = torch.tensor([0, 1, 0, 1, 1, 0])
# target = torch.tensor([0, 1, 1, 0, 1, 0])

# Compute metrics
#sensitivity = sensitivity_metric(preds, target)
#specificity = specificity_metric(preds, target)

# Calculate True Positives, True Negatives, False Positives, False Negatives
TP = torch.sum((preds == 1) & (target == 1)).float()
TN = torch.sum((preds == 0) & (target == 0)).float()
FP = torch.sum((preds == 1) & (target == 0)).float()
FN = torch.sum((preds == 0) & (target == 1)).float()

PCC = (TP + TN) / (TP + TN + FP + FN)
sensitivity = TP / (TP + FN )
specificity = TN / (TN + FP )

print(f"PCC: {PCC.item():.4f}")
print(f"Sensitivity (True Positives): {sensitivity.item():.4f}")
print(f"Specificity (True Negatives): {specificity.item():.4f}")

PCC: 0.9756
Sensitivity (True Positives): 1.0000
Specificity (True Negatives): 0.9494


In [529]:
len(preds), len(target)
preds[0:20], target[0:20]

(tensor([0., 1., 1., 1., 1., 1., 0., 1., 1., 1., 0., 1., 0., 1., 0., 0., 1., 0.,
         1., 0.]),
 tensor([0., 1., 0., 1., 1., 1., 0., 0., 1., 1., 0., 1., 0., 1., 0., 0., 1., 0.,
         1., 0.]))

In [530]:
# AUC Calculation
from sklearn.metrics import roc_auc_score
import numpy as np

#()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()
model = model1  # model 1 or model2, depending on which model you want to evaluate
#()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()()

model1.eval()
with torch.inference_mode():
    preds = torch.round(torch.sigmoid(model(X_test))).squeeze()

# get target / observed response values
target = y_test

preds2 = preds.detach().cpu().tolist()
target2 = target.detach().cpu().tolist()

len(preds), len(target), type(preds), type(preds2), target2[0:5], preds2[0:5]

# 2. Calculate the AUC Score
auc_score = roc_auc_score(target2, preds2)
print(f"Test AUC: {auc_score:.4f}")

Test AUC: 0.9747


# Save the Model

In [622]:
model_dir = future_dir + "/Models"
model_dir

'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/OASES/Data/Future_NO3//Models'

In [623]:
filename = '/torch_model_future_NO3_sw.pth'

# #torch.save(model1.state_dict(), model_dir + filename)
torch.save(model1, model_dir + filename) # saves complete model
